# SageMaker V3 Processing with Instance Preferences (Multi-Instance-Type)

A processing job can name an **ordered list of up to 5 candidate instance types** via the
`Processor`'s `instance_preferences` parameter. SageMaker launches the job on the first
candidate with available capacity and reports the choice as `SelectedInstanceType` /
`SelectedInstanceCount` on the job's `ClusterConfig`.

Training jobs support the same feature — see the
[training example](../training-examples/instance-preferences-example.ipynb).


## Step 1: Setup Session

In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sagemaker_session = Session()
role = get_execution_role()
region = sagemaker_session.boto_region_name

processing_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    instance_type="ml.m5.4xlarge",  # a Step 2 candidate; every candidate must be able to run the image
    image_scope="training",
)


## Step 2: Run a processing job with an ordered list of instance preferences

In [ ]:
from sagemaker.core.processing import Processor

processor = Processor(
    role=role,
    image_uri=processing_image,
    instance_preferences=[
        {"InstanceType": "ml.m5.4xlarge"},
        {"InstanceType": "ml.m5.2xlarge"},
    ],
    instance_count=2,  # applies to whichever preference wins
    volume_size_in_gb=100,
)

processor.run(wait=False, logs=False, job_name="instance-prefs-processing-example")

describe = processor.sagemaker_session.sagemaker_client.describe_processing_job(
    ProcessingJobName="instance-prefs-processing-example"
)
cluster_config = describe["ProcessingResources"]["ClusterConfig"]
print(f"Submitted preferences:   {cluster_config.get('InstancePreferences')}")
print(f"Selected instance type:  {cluster_config.get('SelectedInstanceType')}")
print(f"Selected instance count: {cluster_config.get('SelectedInstanceCount')}")


### Processing with per-preference instance counts

Give **every** preference its own `InstanceCount` instead of the shared top-level
`instance_count` when the candidate types differ in size. Leave `instance_count` unset here.


In [ ]:
per_preference_processor = Processor(
    role=role,
    image_uri=processing_image,
    instance_preferences=[
        # 2 of the larger type...
        {"InstanceType": "ml.m5.4xlarge", "InstanceCount": 2},
        # ...or 4 of the smaller type
        {"InstanceType": "ml.m5.2xlarge", "InstanceCount": 4},
    ],
    volume_size_in_gb=100,
)

per_preference_processor.run(
    wait=False, logs=False, job_name="instance-prefs-processing-per-pref-example"
)


## Notes

- **Backward compatible**: jobs that don't set `instance_preferences` behave exactly as
  before.
- **Not supported with**: local mode and `FrameworkProcessor`. Training plans are
  training-only and do not apply to processing.
- Each instance type may appear only once, and exactly one type is selected per job.
- Selection is based on capacity, not on workload fit: cross-type differences such as
  architecture or GPU memory are not validated, so list only types your job can run on.
- Billing is based on the **selected** instance type and count.
